In [1]:
import importlib
import sys

# Add the directory to sys.path if it's not already there
sys.path.append(r"C:\Users\G0004878\Desktop\TFT_Data\Daily_forecasting_model\Iterations in September\_Iteration_#1_Modelling_after_research\Interpretation")

# Import the module first
import tft_setup
import tft_interpret
import tft_interpret_updated

# Now reload it
importlib.reload(tft_setup)
importlib.reload(tft_interpret)

The StatsForecast module could not be imported. To enable support for the AutoARIMA, AutoETS and Croston models, please consider installing it.
The `XGBoost` module could not be imported. To enable XGBoost support in Darts, follow the detailed instructions in the installation guide: https://github.com/unit8co/darts/blob/master/INSTALL.md
The `XGBoost` module could not be imported. To enable XGBoost support in Darts, follow the detailed instructions in the installation guide: https://github.com/unit8co/darts/blob/master/INSTALL.md


<module 'tft_interpret' from 'c:\\Users\\G0004878\\Desktop\\TFT_Data\\Daily_forecasting_model\\Iterations in September\\_Iteration_#1_Modelling_after_research\\Interpretation\\tft_interpret.py'>

In [2]:
import os 
os.environ["CUDA_LAUNCH_BLOCKING"] = "1"

In [3]:
from tft_setup import load_everything

In [4]:
ctx = load_everything(model_name="daily_tft_negbin_scooters_2026-09-10_18_02_22",work_dir=r"C:\Users\G0004878\Desktop\TFT_Data\Daily_forecasting_model\Iterations in September\_Iteration_#1_Modelling_after_research\Modelling_code",base_dir=r"C:\Users\G0004878\Desktop\TFT_Data\Daily_forecasting_model\Iterations in September\_Iteration_#1_Modelling_after_research\Modelling_code",skip_dead_filter=True)

model folder contents: ['checkpoints', '_model.pth.tar']
model            : daily_tft_negbin_scooters_2026-09-10_18_02_22
ICL / OCL        : 365 / 98
series in manifest after filter : 115,940
series with val strip           : 115,940
history ends     : 2026-08-31  (forecast starts next day)
calendar covers  : 2023-04-01 -> 2026-12-07


In [5]:
from tft_interpret_updated import *

labels = ctx.labels()                                    # one MODEL_FAMILY label per val series
print(pd.Series(labels).value_counts())                  # group sizes: spot tiny groups

idx = stratified_indices(labels, n_per_group=10)         # small first
out = extract_tft_interpretation(
    ctx.model, ctx.val_seq, future_covariates=ctx.val_cov_seq,
    indices=idx, group_labels=labels,
    target_names=[ctx.target_col],
    future_names=ctx.future_covariates,
    static_names=ctx.static_covariates,
)

SPLENDOR+         19487
DESTINI           16220
HF DELUXE         14950
SUPER SPLENDOR    11464
XOOM              10296
GLAMOUR            9565
PLEASURE+          8590
XTREME 160         7411
XTREME 125         6222
PASSION            5304
XPULSE             3878
HF 100             2115
XTREME 250          438
Name: count, dtype: int64
[encoder] raw variable names (63): ['target_0', 'future_covariate_0', 'future_covariate_1', 'future_covariate_2', 'future_covariate_3', 'future_covariate_4'] ...
[decoder] raw variable names (62): ['future_covariate_0', 'future_covariate_1', 'future_covariate_2', 'future_covariate_3', 'future_covariate_4', 'future_covariate_5'] ...
[static] raw variable names (11): ['static_covariate_0', 'static_covariate_1', 'static_covariate_2', 'static_covariate_3', 'static_covariate_4', 'static_covariate_5'] ...


Using bfloat16 Automatic Mixed Precision (AMP)
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
HPU available: False, using: 0 HPUs
You are using a CUDA device ('NVIDIA RTX 2000 Ada Generation') that has Tensor Cores. To properly utilize them, you should set `torch.set_float32_matmul_precision('medium' | 'high')` which will trade-off precision for performance. For more details, read https://pytorch.org/docs/stable/generated/torch.set_float32_matmul_precision.html#torch.set_float32_matmul_precision
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]
c:\Users\G0004878\Desktop\Virtual_environments\darts_gpu\lib\site-packages\pytorch_lightning\trainer\connectors\data_connector.py:425: The 'predict_dataloader' does not have many workers which may be a bottleneck. Consider increasing the value of the `num_workers` argument` to `num_workers=19` in the `DataLoader` to improve performance.


Predicting: |          | 0/? [00:00<?, ?it/s]

Generating TimeSeries:   0%|          | 0/130 [00:00<?, ?it/s]

In [6]:
r = out["ALL"]
print(r["n_series"])
print(r["decoder_importance"].head(10))
print(r["encoder_importance"].head(10))
print(r["static_importance"])
print(r["attention"].sum(axis=0).head()) 

130
MONTH_COS       0.082127
DOW_SIN         0.058809
N+4             0.045824
DOW_COS         0.036239
MONTH_SIN       0.035551
N-4             0.034074
N-3             0.032430
N+10            0.030881
N-10            0.026170
NAG_PANCHAMI    0.026092
dtype: float64
NET_SALES              0.509695
future_covariate_61    0.208705
MONTH_SIN              0.054844
MONTH_COS              0.011562
DOW_SIN                0.010807
MARRIAGE_DAY           0.010421
N-3                    0.007427
N-11                   0.007287
DOM_SIN                0.006662
IS_MONTH_END           0.005248
dtype: float64
MODEL_FAMILY          0.163836
COLOUR                0.162891
MODEL_NAME            0.155757
ZONAL_OFFICE_NAME     0.122390
BRAKE_TYPE            0.109097
DATASET_TYPE          0.077161
WHEEL_TYPE            0.073805
IGNITION_TYPE         0.062263
PARENT_DEALER_CODE    0.044423
DEALER_CITY           0.025032
X_CITY_CATEGORY       0.003346
dtype: float64
h1    1.0
h2    1.0
h3    1.0
h4    1.0


In [7]:
idx = stratified_indices(labels, n_per_group=100)
out = extract_tft_interpretation(ctx.model, ctx.val_seq, future_covariates=ctx.val_cov_seq,
        indices=idx, group_labels=labels, target_names=[ctx.target_col],
        future_names=ctx.future_covariates, static_names=ctx.static_covariates)
save_results(out, "tft_interp")
plot_importance(out["ALL"]["decoder_importance"], title="Decoder importance")
plot_importance(out["ALL"]["encoder_importance"], title="Encoder importance")
plot_attention(out["ALL"]["attention"], title="Attention")

[encoder] raw variable names (63): ['target_0', 'future_covariate_0', 'future_covariate_1', 'future_covariate_2', 'future_covariate_3', 'future_covariate_4'] ...
[decoder] raw variable names (62): ['future_covariate_0', 'future_covariate_1', 'future_covariate_2', 'future_covariate_3', 'future_covariate_4', 'future_covariate_5'] ...
[static] raw variable names (11): ['static_covariate_0', 'static_covariate_1', 'static_covariate_2', 'static_covariate_3', 'static_covariate_4', 'static_covariate_5'] ...


Using bfloat16 Automatic Mixed Precision (AMP)
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
HPU available: False, using: 0 HPUs
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]
c:\Users\G0004878\Desktop\Virtual_environments\darts_gpu\lib\site-packages\pytorch_lightning\trainer\connectors\data_connector.py:425: The 'predict_dataloader' does not have many workers which may be a bottleneck. Consider increasing the value of the `num_workers` argument` to `num_workers=19` in the `DataLoader` to improve performance.


Predicting: |          | 0/? [00:00<?, ?it/s]

RuntimeError: CUDA error: device-side assert triggered
Compile with `TORCH_USE_CUDA_DSA` to enable device-side assertions.
